# LangGraph RAG Agent with pgvector

This notebook builds a **Retrieval-Augmented Generation (RAG) agent** using LangGraph and pgvector,
with a real-world document ingestion pipeline:

1. **PDF → Markdown** — convert research papers using [MarkItDown](https://github.com/microsoft/markitdown)
2. **Chunk** — split markdown into retrieval-friendly pieces
3. **Embed & Store** — store embeddings in pgvector
4. **Agent** — LangGraph agent decides when to retrieve and synthesizes answers

```mermaid
graph LR
    PDF["📄 PDFs<br/>(data/)"] -->|MarkItDown| MD["📝 Markdown"]
    MD -->|RecursiveCharacterTextSplitter| Chunks["📦 Chunks"]
    Chunks -->|databricks-gte-large-en| PG["🐘 pgvector"]
    PG -->|retriever tool| Agent["🤖 LangGraph Agent"]
    Agent -->|PostgresSaver| Memory["💾 Persistent Memory"]
```

**What makes this different from a basic RAG chain:**
- Real document ingestion pipeline (PDF → Markdown → chunks → embeddings)
- The agent decides whether retrieval is needed (not every question requires it)
- It can search multiple times with different queries to gather more context
- The graph is persistent — uses PostgresSaver so conversations survive restarts

**Source papers (in `data/`):**
- Retrieval-Augmented Generation (Lewis et al., 2020)
- AutoGen: Enabling Next-Gen LLM Apps via Multi-Agent Conversations
- Efficient Memory Management for LLM Serving with PagedAttention
- Transformer Explainer: Interactive Learning of Text-Generative Models

**Prerequisites:**
- pgvector pod running (`podman pod start pgvector-pod`)
- MLflow UI running (`mlflow ui --port 5000`)
- `.env` file configured
- `markitdown` installed (`uv pip install markitdown`)

## Step 1 — Bootstrap & Connect

In [ ]:
%run ../../langchain_common.py

In [ ]:
print(f"Host:       {DATABRICKS_HOST}")
print(f"Model:      {DATABRICKS_MODEL}")
print(f"PGVector:   {pgvectordb_conn}")

## Step 2 — Convert PDFs to Markdown with MarkItDown

[MarkItDown](https://github.com/microsoft/markitdown) converts PDFs (and many other formats) to clean Markdown text.
This gives us structured, readable content that chunks much better than raw PDF extraction.

We convert each paper, then wrap the output as a LangChain `Document` with source metadata.

In [ ]:
import os
from pathlib import Path
from markitdown import MarkItDown
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_DIR = Path("data")
md_converter = MarkItDown()

# ── Step 2a: Convert PDFs to Markdown ─────────────────────────────────────────
docs = []
for pdf_path in sorted(DATA_DIR.glob("*.pdf")):
    print(f"Converting: {pdf_path.name}")
    result = md_converter.convert(str(pdf_path))
    doc = Document(
        page_content=result.text_content,
        metadata={"source": pdf_path.name},
    )
    docs.append(doc)
    print(f"  → {len(result.text_content):,} chars")

print(f"\nConverted {len(docs)} PDF(s) to Markdown")

### Step 2b — Chunk the Markdown

Split each document into retrieval-friendly chunks. The `RecursiveCharacterTextSplitter`
with tiktoken encoding gives us token-aware splits that respect paragraph boundaries.

In [ ]:
# ── Step 2b: Chunk into retrieval-friendly sizes ──────────────────────────────
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50,
)
doc_splits = splitter.split_documents(docs)

print(f"Total chunks: {len(doc_splits)}")
print(f"\nChunks per source:")
from collections import Counter
for source, count in Counter(d.metadata["source"] for d in doc_splits).items():
    print(f"  {source}: {count} chunks")

print(f"\nSample chunk (from {doc_splits[0].metadata['source']}):")
print(doc_splits[0].page_content[:300] + "...")

## Step 3 — Store Embeddings in pgvector

Embed all chunks using the Databricks embedding model (`databricks-gte-large-en`, 1024-dim)
and store them in a pgvector collection. This completes the ingestion pipeline:

**PDF → MarkItDown → Chunks → Embeddings → pgvector**

In [ ]:
COLLECTION_NAME = "langgraph_rag_demo"

vectorstore = PGVector.from_documents(
    documents=doc_splits,
    embedding=databricks_embeddings,
    collection_name=COLLECTION_NAME,
    connection=pgvectordb_conn,
    use_jsonb=True,
    pre_delete_collection=True,  # fresh start each run
)

# Quick test — similarity search
test_results = vectorstore.similarity_search_with_score("What is RAG?", k=2)
for doc, score in test_results:
    print(f"score={score:.4f} | {doc.page_content[:100]}...")

## Step 4 — Define the Retrieval Tool

We wrap the vector store retriever as a LangChain tool so the LangGraph agent can
decide when to call it. The tool returns the top-k relevant chunks for a given query.

In [ ]:
from langchain_core.tools import tool

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base of AI research papers for information about
    LLMs, RAG, multi-agent systems, transformers, and attention mechanisms.
    Use this tool when the user asks factual questions about these topics.

    Args:
        query: the search query describing what information you need
    """
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant documents found in the knowledge base."

    results = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown")
        results.append(f"[{i}] (source: {source})\n{doc.page_content}")

    return "\n\n".join(results)


# Verify the tool works
print(search_knowledge_base.invoke("What is retrieval augmented generation?")[:500])

## Step 5 — Build the LangGraph RAG Agent

The agent uses:
- A **system prompt** that instructs it to use the retrieval tool for factual questions
- **`tools_condition`** for automatic routing (tool calls vs. final answer)
- **PostgresSaver** for persistent conversation memory across restarts

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import HumanMessage, SystemMessage

RAG_SYSTEM_PROMPT = """You are a knowledgeable AI research assistant with access to a \
knowledge base of research papers about Large Language Models, Retrieval-Augmented \
Generation, multi-agent systems, and transformer architectures.

Rules:
1. When the user asks factual questions about LLMs, RAG, transformers, attention, \
   multi-agent systems, or related AI topics, ALWAYS use the search_knowledge_base \
   tool first to find relevant information from the papers.
2. You may search multiple times with different queries to gather comprehensive context.
3. Base your answers on the retrieved documents. Cite the source paper when possible.
4. If the knowledge base doesn't have relevant information, say so honestly and provide \
   your best general knowledge answer.
5. For non-factual requests (greetings, math, opinions), respond directly without searching.
"""

tools = [search_knowledge_base]
llm_with_tools = llm_noreason.bind_tools(tools)


def assistant(state: MessagesState):
    messages = [SystemMessage(content=RAG_SYSTEM_PROMPT)] + state["messages"]
    return {"messages": [llm_with_tools.invoke(messages)]}


# Build the graph
builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

# Compile with persistent Postgres checkpointer
pg_checkpointer = create_pg_checkpointer()
graph = builder.compile(checkpointer=pg_checkpointer)

print("RAG agent compiled with pgvector retrieval + Postgres checkpointer")

## Step 6 — Test the Agent

Let's ask questions that require retrieval and some that don't, to see the agent
decide when to use the knowledge base.

In [ ]:
# Start a conversation thread
config = make_thread_config()
print(f"Thread: {config['configurable']['thread_id']}\n")

# Question 1: Requires retrieval (from the RAG paper)
result = graph.invoke(
    {"messages": [HumanMessage(content="How does RAG combine retrieval with generation? What architecture does the original paper propose?")]},
    config=config,
)
print("Q: How does RAG combine retrieval with generation?")
print(f"A: {result['messages'][-1].content}\n")

In [ ]:
# Question 2: Cross-paper question (AutoGen + PagedAttention)
result = graph.invoke(
    {"messages": [HumanMessage(content="What is PagedAttention and how does it improve LLM serving efficiency?")]},
    config=config,
)
print("Q: What is PagedAttention and how does it improve LLM serving efficiency?")
print(f"A: {result['messages'][-1].content}\n")

In [ ]:
# Question 3: Does NOT require retrieval — agent should answer directly
result = graph.invoke(
    {"messages": [HumanMessage(content="What is 42 * 17?")]},
    config=config,
)
print("Q: What is 42 * 17?")
print(f"A: {result['messages'][-1].content}")
print("\n(The agent should answer directly without searching the knowledge base)")

## Step 7 — Inspect the Agent's Reasoning

Let's look at the full message trace for a query to see _how_ the agent
decides to retrieve and synthesize.

In [ ]:
# New thread for a clean trace
trace_config = make_thread_config()

result = graph.invoke(
    {"messages": [HumanMessage(content="How does AutoGen enable multi-agent conversations and what design patterns does it use?")]},
    config=trace_config,
)

print("=" * 60)
print("Full message trace:")
print("=" * 60)
for msg in result["messages"]:
    msg.pretty_print()
    print()

## Step 8 — Verify Persistence

Because we used PostgresSaver, the conversation state is stored in Postgres.
We can reload a previous thread and continue the conversation.

In [ ]:
# Retrieve the state from the first conversation thread
state = graph.get_state(config)
print(f"Thread: {config['configurable']['thread_id']}")
print(f"Messages in thread: {len(state.values['messages'])}")
print(f"\nLast message: {state.values['messages'][-1].content[:200]}...")

# Continue the conversation
result = graph.invoke(
    {"messages": [HumanMessage(content="Can you summarize everything we discussed?")]},
    config=config,
)
print(f"\nSummary: {result['messages'][-1].content}")

## Summary

### Ingestion Pipeline

| Stage | Technology | What it does |
|-------|-----------|--------------|
| 1. Convert | **MarkItDown** | PDF → clean Markdown (preserves headings, tables, structure) |
| 2. Chunk | `RecursiveCharacterTextSplitter` | Split into 300-token pieces with 50-token overlap |
| 3. Embed | `databricks-gte-large-en` (1024-dim) | Convert text chunks to dense vectors |
| 4. Store | **pgvector** (`PGVector`) | Persistent ANN similarity search in Postgres |

### Agent Architecture

| Component | Technology | What it does |
|-----------|-----------|--------------|
| Retrieval tool | `@tool` decorator | Wraps pgvector retriever for agent tool calling |
| Agent graph | **LangGraph** (`StateGraph`) | Decides when to retrieve, synthesizes answers |
| Persistence | **PostgresSaver** | Conversation memory survives restarts |

### Key Takeaways

- **MarkItDown** gives much cleaner text than raw PDF extraction — headings, tables, and
  structure are preserved, leading to better chunks and better retrieval
- The agent _chooses_ when retrieval is needed — for greetings or math it responds directly
- It can search multiple times with refined queries before composing its answer
- **PostgresSaver** means the conversation state persists across kernel restarts